In [1]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 1.8 MB/s eta 0:00:00a 0:00:01


In [ ]:
import json
import pandas as pd
import numpy as np
import re
import spacy
from textstat import flesch_kincaid_grade
from nltk.corpus import stopwords
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

class QueryAnalyzer:
    def __init__(self):
        # Setup
        nltk.download('stopwords', quiet=True)
        self.stop_words = set(stopwords.words('english'))
        self.nlp = spacy.load('en_core_web_sm')
        self.sentence_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    def load_data(self, json_path):
        """Load JSON data and return DataFrame"""
        with open(json_path, 'r') as f:
            data = json.load(f)

        queries = data["analytical_queries"]
        df = pd.DataFrame(queries)
        return df

    # LINGUISTIC VALIDITY METRICS
    def grammatical_completeness(self, doc):
        has_verb = any(tok.pos_ == "VERB" for tok in doc)
        has_noun = any(tok.pos_ == "NOUN" for tok in doc)
        return int(has_verb and has_noun)

    def calculate_linguistic_validity(self, df):
        """Calculate linguistic validity metrics"""
        metrics = []

        for q in df["query"]:
            doc = self.nlp(q)

            G = self.grammatical_completeness(doc)
            R_raw = flesch_kincaid_grade(q)

            metrics.append({
                "query": q,
                "grammatical_completeness": G,
                "readability_score": R_raw
            })

        metrics_df = pd.DataFrame(metrics)
        result = pd.concat([df, metrics_df.drop(columns=["query"])], axis=1)
        return result

    # READABILITY BY DIFFICULTY
    def calculate_readability_by_difficulty(self, df):
        """Calculate readability scores by difficulty level"""
        readability_by_difficulty = df.groupby("difficulty")["readability_score"].agg([
            'count', 'mean', 'std', 'min', 'max'
        ]).round(3)

        return readability_by_difficulty

    # TF-IDF SIMILARITY
    def calculate_tfidf_similarity(self, df):
        """TF-IDF similarity"""
        results_same_mece = []
        results_intra_mece = []

        # Generate TF-IDF matrix for all queries
        vectorizer = TfidfVectorizer(stop_words="english", lowercase=True, max_features=5000)
        tfidf_matrix = vectorizer.fit_transform(df["query"])

        # Add index to dataframe for easy reference
        df = df.copy()
        df["query_index"] = range(len(df))

        # 1. Same Domain + Same MECE similarity
        for idx, row in df.iterrows():
            current_domain = row["domain"]
            current_mece = row["mece_category"]
            current_idx = row["query_index"]

            # Find other queries in same domain + same MECE
            same_group_mask = (df["domain"] == current_domain) & (df["mece_category"] == current_mece) & (df["query_index"] != current_idx)
            same_group_indices = df[same_group_mask]["query_index"].tolist()

            if len(same_group_indices) > 0:
                # Calculate similarity with all other queries in same group
                similarities = []
                for other_idx in same_group_indices:
                    sim = cosine_similarity(
                        tfidf_matrix[current_idx:current_idx+1],
                        tfidf_matrix[other_idx:other_idx+1]
                    )[0][0]
                    similarities.append(sim)

                if similarities:
                    results_same_mece.append({
                        "query": row["query"],
                        "domain": current_domain,
                        "mece_category": current_mece,
                        "same_group_similarity": np.mean(similarities),
                        "comparisons_count": len(similarities)
                    })

        # 2. Intra MECE similarity (cross MECE within same domain)
        for idx, row in df.iterrows():
            current_domain = row["domain"]
            current_mece = row["mece_category"]
            current_idx = row["query_index"]

            # Find queries in same domain but DIFFERENT MECE categories
            intra_mece_mask = (df["domain"] == current_domain) & (df["mece_category"] != current_mece)
            intra_mece_indices = df[intra_mece_mask]["query_index"].tolist()

            if len(intra_mece_indices) > 0:
                # Calculate similarity with queries from other MECE categories
                similarities = []
                for other_idx in intra_mece_indices:
                    sim = cosine_similarity(
                        tfidf_matrix[current_idx:current_idx+1],
                        tfidf_matrix[other_idx:other_idx+1]
                    )[0][0]
                    similarities.append(sim)

                if similarities:
                    results_intra_mece.append({
                        "query": row["query"],
                        "domain": current_domain,
                        "current_mece": current_mece,
                        "compared_mece_categories": df[intra_mece_mask]["mece_category"].nunique(),
                        "intra_mece_similarity": np.mean(similarities),
                        "comparisons_count": len(similarities)
                    })

        return pd.DataFrame(results_same_mece), pd.DataFrame(results_intra_mece)

    # SEMANTIC SIMILARITY
    def calculate_semantic_similarity(self, df):
        """Semantic similarity"""
        results_same_mece = []
        results_intra_mece = []

        print("Generating embeddings...")

        # Generate embeddings for all queries
        embeddings = self.sentence_model.encode(
            df["query"].tolist(),
            normalize_embeddings=False,
            show_progress_bar=False
        )

        df = df.copy()
        df["embedding"] = list(embeddings)
        df["query_index"] = range(len(df))

        # 1. Same Domain + Same MECE similarity
        for idx, row in df.iterrows():
            current_domain = row["domain"]
            current_mece = row["mece_category"]
            current_embedding = row["embedding"]
            current_idx = row["query_index"]

            # Find other queries in same domain + same MECE
            same_group_mask = (df["domain"] == current_domain) & (df["mece_category"] == current_mece) & (df["query_index"] != current_idx)
            same_group_df = df[same_group_mask]

            if len(same_group_df) > 0:
                # Calculate similarity with all other queries in same group
                similarities = []
                for _, other_row in same_group_df.iterrows():
                    other_embedding = other_row["embedding"]
                    # Manual cosine similarity
                    dot_product = np.dot(current_embedding, other_embedding)
                    norm1 = np.linalg.norm(current_embedding)
                    norm2 = np.linalg.norm(other_embedding)

                    if norm1 > 0 and norm2 > 0:
                        cos_sim = dot_product / (norm1 * norm2)
                        similarities.append(cos_sim)

                if similarities:
                    results_same_mece.append({
                        "query": row["query"],
                        "domain": current_domain,
                        "mece_category": current_mece,
                        "same_group_semantic_similarity": np.mean(similarities),
                        "semantic_comparisons_count": len(similarities),
                        "min_semantic_similarity": np.min(similarities),
                        "max_semantic_similarity": np.max(similarities)
                    })

        # 2. Intra MECE similarity (cross MECE within same domain)
        for idx, row in df.iterrows():
            current_domain = row["domain"]
            current_mece = row["mece_category"]
            current_embedding = row["embedding"]

            # Find queries in same domain but DIFFERENT MECE categories
            intra_mece_mask = (df["domain"] == current_domain) & (df["mece_category"] != current_mece)
            intra_mece_df = df[intra_mece_mask]

            if len(intra_mece_df) > 0:
                # Calculate similarity with queries from other MECE categories
                similarities = []
                for _, other_row in intra_mece_df.iterrows():
                    other_embedding = other_row["embedding"]
                    # Manual cosine similarity
                    dot_product = np.dot(current_embedding, other_embedding)
                    norm1 = np.linalg.norm(current_embedding)
                    norm2 = np.linalg.norm(other_embedding)

                    if norm1 > 0 and norm2 > 0:
                        cos_sim = dot_product / (norm1 * norm2)
                        similarities.append(cos_sim)

                if similarities:
                    results_intra_mece.append({
                        "query": row["query"],
                        "domain": current_domain,
                        "current_mece": current_mece,
                        "compared_mece_categories": intra_mece_df["mece_category"].nunique(),
                        "intra_mece_semantic_similarity": np.mean(similarities),
                        "semantic_comparisons_count": len(similarities),
                        "min_intra_similarity": np.min(similarities),
                        "max_intra_similarity": np.max(similarities)
                    })

        return pd.DataFrame(results_same_mece), pd.DataFrame(results_intra_mece)

    # COMPREHENSIVE ANALYSIS
    def analyze_queries(self, json_path):
        """Run complete analysis with the methodology"""
        print(f"Loading data from {json_path}...")
        df = self.load_data(json_path)

        print("\n=== LINGUISTIC VALIDITY ANALYSIS ===")
        linguistic_df = self.calculate_linguistic_validity(df)

        grammar_ok = linguistic_df["grammatical_completeness"].mean() * 100
        print(f"Grammar OK: {grammar_ok:.1f}%")

        print("\n=== READABILITY BY DIFFICULTY LEVEL ===")
        readability_by_difficulty = self.calculate_readability_by_difficulty(linguistic_df)
        print(readability_by_difficulty)

        print("\n=== TF-IDF SIMILARITY ===")
        tfidf_same, tfidf_intra = self.calculate_tfidf_similarity(linguistic_df)

        print("Same Domain + Same MECE:")
        if len(tfidf_same) > 0:
            print(f"  Queries analyzed: {len(tfidf_same)}")
            print(f"  Avg Similarity: {tfidf_same['same_group_similarity'].mean():.3f}")
            print(f"  Range: {tfidf_same['same_group_similarity'].min():.3f} - {tfidf_same['same_group_similarity'].max():.3f}")

        print("Intra MECE (Cross MECE within Domain):")
        if len(tfidf_intra) > 0:
            print(f"  Queries analyzed: {len(tfidf_intra)}")
            print(f"  Avg Cross-MECE Similarity: {tfidf_intra['intra_mece_similarity'].mean():.3f}")
            print(f"  Range: {tfidf_intra['intra_mece_similarity'].min():.3f} - {tfidf_intra['intra_mece_similarity'].max():.3f}")

        print("\n=== SEMANTIC SIMILARITY ===")
        semantic_same, semantic_intra = self.calculate_semantic_similarity(linguistic_df)

        print("Same Domain + Same MECE:")
        if len(semantic_same) > 0:
            print(f"  Queries analyzed: {len(semantic_same)}")
            print(f"  Avg Semantic Similarity: {semantic_same['same_group_semantic_similarity'].mean():.3f}")
            print(f"  Range: {semantic_same['min_semantic_similarity'].mean():.3f} - {semantic_same['max_semantic_similarity'].mean():.3f}")

        print("Intra MECE (Cross MECE within Domain):")
        if len(semantic_intra) > 0:
            print(f"  Queries analyzed: {len(semantic_intra)}")
            print(f"  Avg Cross-MECE Semantic Similarity: {semantic_intra['intra_mece_semantic_similarity'].mean():.3f}")
            print(f"  Range: {semantic_intra['min_intra_similarity'].mean():.3f} - {semantic_intra['max_intra_similarity'].mean():.3f}")

        # Save results
        self.save_results(linguistic_df, readability_by_difficulty, tfidf_same, tfidf_intra, semantic_same, semantic_intra)

        return linguistic_df

    def save_results(self, linguistic_df, readability_by_difficulty, tfidf_same, tfidf_intra, semantic_same, semantic_intra):
        """Save all results to CSV files"""
        linguistic_df.to_csv("linguistic_validity_analysis(2).csv", index=False)
        readability_by_difficulty.to_csv("readability_by_difficulty(2).csv")
        tfidf_same.to_csv("tfidf_same_mece_similarity2.csv", index=False)
        tfidf_intra.to_csv("tfidf_intra_mece_similarity2.csv", index=False)
        semantic_same.to_csv("semantic_same_mece_similarity2.csv", index=False)
        semantic_intra.to_csv("semantic_intra_mece_similarity2.csv", index=False)

        print("\n Results saved:")
        print("  - linguistic_validity_analysis2.csv")
        print("  - readability_by_difficulty2.csv")
        print("  - tfidf_same_mece_similarity2.csv")
        print("  - tfidf_intra_mece_similarity2.csv")
        print("  - semantic_same_mece_similarity2.csv")
        print("  - semantic_intra_mece_similarity2.csv")

# === USAGE ===
if __name__ == "__main__":
    analyzer = QueryAnalyzer()

    OUR_JSON_FILE = "test1.json"  # ← CHANGE THIS

    try:
        results = analyzer.analyze_queries(OUR_JSON_FILE)

    except FileNotFoundError:
        print(f"\n Error: File '{OUR_JSON_FILE}' not found!")
    except Exception as e:
        print(f"\n An error occurred: {e}")
        import traceback
        traceback.print_exc()